<a href="https://colab.research.google.com/github/evgeneachebakova-art/2/blob/main/%D0%B7%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Уровень 4. Архитектор (задачи 25–30)
Что изучается: паттерны Factory и Observer, принцип SRP из SOLID, выбор между композицией и наследованием, копирование объектов, утиная типизация.

Задача 25 (паттерн Factory). Зоопарк регулярно пополняется разными классами животных — млекопитающими, птицами, рептилиями. Создавать их напрямую через конструкторы неудобно: код разрастается, легко ошибиться. Создайте класс AnimalFactory с методом create(kind, **kwargs), который по строке "mammal", "bird" или "reptile" создаёт соответствующий объект и возвращает его. Неизвестный тип должен вызывать ValueError. Продемонстрируйте работу фабрики: создайте по одному животному каждого класса через фабрику. Объясните, почему фабрика удобнее прямого вызова конструкторов и как этот паттерн связан с принципом Open/Closed (см. задачу 35).

In [ ]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def info(self):
        return f" {self.__class__.__name__}: Имя - {self.name}, Возраст - {self.age}"

class Mammal(Animal):
    def __init__(self, name, age, fur_color):
        super().__init__(name, age)
        self.fur_color = fur_color

    def info(self):
        return f" {super().info()}, Цвет шерсти - {self.fur_color}"

class Bird(Animal):
    def __init__(self, name, age,  wingspan):
        super().__init__(name, age)
        self.wingspan = wingspan

    def info(self):
        return f"{super().info()}, Размах крыльев - {self.wingspan}"

class Reptile(Animal):
    def __init__(self, name, age, is_venomous):
        super().__init__(name, age)
        self.is_venomous = is_venomous

    def info(self):
        venom = "ядовитый" if self.is_venomous else "неядовитый"
        return f"{super().info()}, {venom}"

class AnimalFactory:
    @staticmethod
    def create(kind, **kwargs):
        if kind == "Mammal":
            return Mammal(**kwargs)
        elif kind == "Bird":
            return Bird(**kwargs)
        elif kind == "Reptile":
            return Reptile(**kwargs)
        else:
            raise ValueError(f"Неизвестный тип животного: {kind}")

#Демонстрация

mammal = AnimalFactory.create(kind="Mammal", name="Симба", age=7, fur_color="orange")
bird = AnimalFactory.create(kind= "Bird", name= "Кеша", age= 8, wingspan= 3)
reptile = AnimalFactory.create(kind= "Reptile", name= "Григорий", age= 9, is_venomous= True)

print(mammal.info())
print(bird.info())
print(reptile.info())

try:
    AnimalFactory.create(kind="fish", name="Немо", age=1)
except ValueError as e:
    print("\nОшибка:", e)

"""
Почему фабрика удобнее прямого вызова конструкторов?
Вместо того чтобы разбрасывать по коду условия if/elif для выбора нужного класса, мы
собираем эту логику в одном месте. Пользователю фабрики не нужно знать
о существовании всех классов-наследников, он просто заказывает животное
по строковому имени, передавая нужные аргументы через **kwargs.

Как этот паттерн связан с принципом Open/Closed (из SOLID)?
Принцип Open/Closed гласит: классы должны быть открыты для расширения,
но закрыты для изменения. Фабрика реализует этот принцип: чтобы добавить
новый тип животного (например, Fish), не меняется существующий код
клиентов, которые используют фабрику, а просто расширяется сама фабрика,
добавлением новой ветки elif. Старый код при этом не ломается и не требует
правки.
"""



Задача 26 (паттерн Observer). Посетители зоопарка хотят получать уведомления, когда рождается детёныш или открывается новое шоу. Реализуйте паттерн «Наблюдатель»: класс Zoo хранит список подписчиков (Visitor) и при появлении важного события уведомляет всех через метод notify(). Продемонстрируйте, что при появлении события все посетители получают уведомление. Дополнительно реализуйте unsubscribe() и покажите, что после отписки посетитель больше не получает уведомлений. Объясните, где в реальных проектах используется этот паттерн (GUI, событийные системы, мессенджеры, pub/sub) и как он помогает избежать жёсткой связи между объектами.

In [ ]:
class Visitor:
    def __init__(self, name):
        self.name = name

    def update(self, event_message):
        print(f"{self.name} получил уведомление: {event_message}")

class Zoo:
    def __init__(self, name):
        self.name = name
        self.subscribers = []

    def subscribe(self, visitor):
        if visitor not in self.subscribers:
            self.subscribers.append(visitor)
            print(f"{visitor.name} подписался на уведомления")

    def unsubscribe(self, visitor):
        if visitor in self.subscribers:
            self.subscribers.remove(visitor)
            print(f"{visitor.name} отписался от уведомлений ")

    def  notify (self, event_message):
        print(f"В зоопарке {self.name}: {event_message}")
        for visitor in self.subscribers:
            visitor.update(event_message)

#Демонстрация

zoo = Zoo("Дом животных")

anna = Visitor("Anna")
jenny = Visitor("Jenny")
max = Visitor("Max")

zoo.subscribe(anna)
zoo.subscribe(jenny)
zoo.subscribe(max)

zoo.unsubscribe(anna)

zoo.notify("Родился новый львенок")


"""
Где используется паттерн Observer в реальных проектах?
1. GUI (графические интерфейсы): кнопки и поля ввода подписываются на
   события мыши или клавиатуры.
2. Мессенджеры и чаты: пользователи подписываются на каналы или группы
   и получают новые сообщения.
3. Архитектура Pub/Sub (Publish/Subscribe): системы очередей сообщений
   (например, RabbitMQ или Kafka), где микросервисы реагируют на события
   друг друга, не зная об их существовании напрямую.
4. Реактивное программирование: фреймворки вроде React или Vue.js
   обновляют интерфейс при изменении данных.

Как он помогает избежать жёсткой связи?
Издатель (Zoo) не знает конкретных классов подписчиков и не зависит от
них. Он хранит только абстрактный список объектов и вызывает у них
стандартный метод update. Благодаря этому можно легко добавлять новых
подписчиков или удалять их,
не изменяя ни единой строчки кода в классе Издателя. Это реализует
принцип открытости/закрытости и делает систему гибкой.
"""




Задача 27 (принцип SOLID — SRP). Класс Zoo со временем разросся: он и хранит животных, и регистрирует кормления, и отправляет уведомления, и формирует отчёты. Это нарушает принцип единственной ответственности (SRP) — у класса должна быть одна причина для изменения. Разбейте класс Zoo на несколько классов: AnimalStorage (хранение животных и поиск по кличке), FeedingService (регистрация кормлений, ведение учёта), NotificationService (уведомления). Продемонстрируйте работу собранных сервисов на сценарии «смотритель покормил животное». Объясните, почему это лучше одного большого класса, как понять, что у класса «одна причина для изменения», и приведите примеры «классов-богов» из реальной жизни.

In [ ]:
class Animal:
    def __init__(self, name, species):
        self.name = name
        self.species = species

class AnimalStorage:
    def __init__(self):
        self.animals = {}

    def add(self, animal):
        self.animals[animal.name] = animal
        print(f" {animal.name} добавлен в хранилище")

    def find(self, name):
        if name not in self.animals:
            raise ValueError(f"Животное '{name}' не найдено")
        else:
            return self.animals[name]

class FeedingService:
    def __init__(self):
        self.feeding_log = []

    def feed(self, animal, food):
        self.feeding_log.append({"animal": animal.name, "food": food})

        print (f" {animal.name} покормлен: {food}")

    def info(self):
        return self.feeding_log

class NotificationService:

    def notify(self, event_message):
        print(f"Уведомление {event_message}")

#Демонстрация на сценарии «смотритель покормил животное»
storage = AnimalStorage()
feeding = FeedingService()
notifier = NotificationService()

animal = Animal("Симба", "Лев")

storage.add(animal)
found_animal = storage.find("Симба")


feeding.feed(found_animal, "Мясо")


notifier.notify("Животное Симба покормили")

print("\nИстория кормлений:", feeding.info())

"""
Почему это лучше одного большого класса?
Один большой класс (Zoo) нарушает SRP, так как отвечает и за хранение,
и за кормление, и за уведомления. Если изменится формат отчётов, придётся
править класс Zoo, рискуя сломать логику хранения. Разбивка на сервисы
делает код модульным: каждый класс решает одну задачу, его легко читать,
тестировать и изменять независимо от остальных.

Как понять, что у класса «одна причина для изменения»?
Нужно задать вопрос: «Из-за каких изменений в требованиях мне придётся
переписывать этот класс?» Если причин несколько (например, изменился способ
хранения данных И формат уведомлений), то SRP нарушен. Если причина одна
(например, «изменился формат журнала кормлений»), то SRP соблюдён.

Примеры «классов-богов» (God Classes) из реальной жизни:
1. God Controller в веб-разработке: один огромный класс, который принимает
   HTTP-запросы, работает с базой данных, отправляет email и рендерит HTML.
2. Классы Utils/Helpers: «свалка» из сотен несвязанных статических методов,
   куда разработчики скидывают всё, что не подошло в другие классы.
3. Модель (Model) в паттерне MVC без разделения: содержит и бизнес-логику,
   и валидацию, и SQL-запросы, и форматирование данных для вывода.
"""





Задача 28 (композиция vs наследование). Морские животные (SeaAnimal) похожи на обычных животных, но имеют дополнительные поля — глубина обитания, солёность воды. Реализуйте одну и ту же задачу двумя способами: через наследование (SeaAnimal(Animal) — переопределяет конструктор и добавляет свои поля) и через композицию (SeaAnimal содержит объект Animal внутри себя как поле self.animal). Сравните плюсы и минусы каждого подхода: наследование даёт переиспользование кода, но создаёт жёсткую связь; композиция гибче, но требует больше кода. Объясните, когда что выбирать, и приведите примеры из реальных проектов и стандартной библиотеки Python.

In [ ]:
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def info(self):
        return f" {self.__class__.__name__}: Имя - {self.name}, Возраст - {self.age}"

#через наследование
class SeaAnimal(Animal):
    def __init__(self, name, age, depth_of_habitation, water_salinity):
        super().__init__(name, age)
        self.depth_of_habitation = depth_of_habitation
        self.water_salinity = water_salinity

    def info(self):
       return f"{super().info()}, глубина обитания: {self.depth_of_habitation}, соленость воды: {self.water_salinity} %"

#через композицию
class SeaAnimalComposed:
    def __init__(self, animal, depth_of_habitation, water_salinity):
        self.animal = animal
        self.depth_of_habitation = depth_of_habitation
        self.water_salinity = water_salinity
    def info(self):
        return f"{self.animal.info()}, глубина обитания: {self.depth_of_habitation}, соленость воды: {self.water_salinity} %"

#Демонстрация для наследования
seaanimal = SeaAnimal("Акула", 3, 500, 35 )
print(seaanimal.info())

#Демонстрация для композиции
base_seaanimal = Animal("Акула", 3)
seaanimal2 = SeaAnimalComposed(base_seaanimal,500, 35)
print(seaanimal2.info())

"""
Сравнение подходов:
Наследование даёт автоматическое переиспользование кода и требует меньше
бойлерплейта, но создаёт жёсткую связь: изменение в базовом классе может
непредсказуемо сломать наследников. Композиция обеспечивает слабую связь,
позволяет динамически подменять компоненты и избегать проблем множественного
наследования, но требует больше кода из-за явного делегирования вызовов.

Когда что выбирать:
Наследование стоит использовать для строгих отношений (IS-A) в
неглубоких иерархиях (например, class MyValueError(ValueError)). Композицию
 следует предпочитать в сложных системах, при работе с
чужим кодом или когда поведение нужно комбинировать динамически..
"""




Задача 29 (копирование объектов). В зоопарке нужно создавать копии карточек животных — например, для архивного хранения. При этом у животного есть вложенный объект Enclosure, и важно понимать, копируется ли он вместе с животным или остаётся общим. Реализуйте в классе Animal методы __copy__ и __deepcopy__. Создайте животное с вложенным объектом Enclosure и продемонстрируйте разницу между поверхностным (copy.copy) и глубоким (copy.deepcopy) копированием: при поверхностном — вольер общий (shallow.enclosure is animal.enclosure → True), при глубоком — свой (deep.enclosure is animal.enclosure → False). Объясните, когда какой тип копирования использовать и что делать с циклическими ссылками.

In [ ]:
import copy
class Enclosure:
    def __init__(self, name):
        self.name = name

    def info(self):
        return f"Enclosure '{self.name}'"

class Animal:
    def __init__(self, name, age, enclosure):
        self.name = name
        self.age = age
        self.enclosure = enclosure

#Поверхностное копирование
    def __copy__(self):
        # 1 Создаем новый пустой объект того же класса
        cls = self.__class__
        new_obj = cls.__new__(cls)

        # 2 Копирование
        new_obj.__dict__.update(self.__dict__)
        return new_obj

#Глубокое копирование
    def __deepcopy__(self, memo):
        # 1 Создаем новый пустой объект
        cls = self.__class__
        new_obj = cls.__new__(cls)
        # 2 Защита от циклов
        memo[id(self)] = new_obj
        # 3 Копирование
        for key, value in self.__dict__.items():
            setattr(new_obj, key, copy.deepcopy(value, memo))
        return new_obj

    def info(self):
        return f"Animal('{self.name}', '{self.age}', {self.enclosure.info()})"

#Демонстрация

print("оригинал")
enclosure1 = Enclosure("Вольер Африка")
animal = Animal("Симба", 3, enclosure1)
print(f"Оригинал: {animal.info()}")
print(f"ID вольера оригинала: {id(animal.enclosure)}\n")

print("Создаем поверхностную копию")
animal2 = copy.copy(animal)
print(f"п.копия:  {animal2.info()}")
print(f"ID п.копии: {id(animal2.enclosure)}")

print("Создаем глубокую копию")
animal3 = copy.deepcopy(animal)
print(f"Deep:     {animal3.info()}")
print(f"ID г.копии:    {id(animal3.enclosure)}")

print("Проверка независимости:")
animal3.enclosure.name = "Новый Вольер"
print(f"Имя вольера у г.копии: {animal3.enclosure.name}")
print(f"Имя вольера у оригинала: {animal.enclosure.name} ")

"""
Когда использовать поверхностное копирование (copy.copy)?
Когда вложенные объекты неизменяемы (строки, числа, кортежи) или когда намеренно нужно,
чтобы копия и оригинал разделяли одни и те же
вложенные объекты (например, общий кэш или конфигурацию).
Это быстрее и потребляет меньше памяти.

Когда использовать глубокое копирование (copy.deepcopy):
Когда вложенные объекты изменяемы (списки, словари, пользовательские
классы) и нужно, чтобы копия была полностью независимой от оригинала.
Любые изменения в копии не должны затрагивать оригинал.

Что делать с циклическими ссылками:
Модуль copy автоматически обрабатывает их благодаря словарю memo.
При реализации __deepcopy__ критически важно сразу после создания
нового объекта добавить его в memo: memo[id(self)] = new_obj.
Это предотвращает бесконечную рекурсию, так как при повторной встрече
с тем же объектом copy.deepcopy вернет уже созданную копию из memo,
а не начнет копировать его заново.
"""





Задача 30 (утиная типизация). В зоопарке есть разные объекты, у которых есть метод describe() — животные, вольеры, секции, а может быть, и что-то ещё, что появится в будущем. Напишите функцию describe_item(item), которая вызывает метод describe() у любого объекта. Продемонстрируйте, что функция работает не только с Animal, но и с любым другим классом, у которого есть describe() (например, классом Section), — без наследования от общего родителя. Объясните принцип утиной типизации: «если это выглядит как утка, плавает как утка и крякает как утка — это утка». Расскажите, как этот принцип отличает Python от языков со строгой типизацией (Java, C++) и какие у него минусы (ошибка видна только во время выполнения).

In [ ]:
class Animal:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    def describe(self):
        return f"Животное: {self.name} - {self.species}"


class Enclosure:
    def __init__(self, name, capacity):
        self.name = name
        self.capacity = capacity

    def describe(self):
        return f"Вольер: {self.name} (вместимость: {self.capacity})"


class Section:
    def __init__(self, zone, curator):
        self.zone = zone
        self.curator = curator

    def describe(self):
        return f"Секция: зона {self.zone}, куратор: {self.curator}"

def describe_item(item):
    print(item.describe())

#Демонстрация

simba = Animal("Симба", "Лев")
africa = Enclosure("Африка", 10)
savanna = Section("A1", "Иван Иванов")

describe_item(simba)
describe_item(africa)
describe_item(savanna)

try:
    describe_item("аааааааа")
except AttributeError as e:
    print(f" Ошибка: {e}")

"""
Принцип утиной типизации
«Если это выглядит как утка, плавает как утка и крякает как утка — это утка».
В Python не важно, к какому классу принадлежит объект. Важно только,
какие методы у него есть. Если у объекта есть нужный метод (например,
describe()), значит, его можно использовать без проверки типа и
без наследования от общего родителя.

Отличие от языков со строгой типизацией (Java, C++)
В Java/C++ функция может принимать только объекты определённого класса
или реализующие конкретный интерфейс. Компилятор проверяет типы ДО
запуска программы. В Python проверка происходит ВО ВРЕмя выполнения, т.е.
если у объекта нет нужного метода, программа упадёт с AttributeError
только в момент вызова.

Минусы утиной типизации:
1. Ошибки видны только во время выполнения.
2. Сложнее понять, какие типы принимает функция — нужно читать документацию
   или искать классы с нужным методом.
3. IDE сложнее давать подсказки, так как тип объекта неизвестен заранее.
"""

Уровень 5. Эксперт (задачи 31–35)
Что изучается: протоколы итерации и индексации (__iter__, __next__, __getitem__, __setitem__), контекстные менеджеры (__enter__, __exit__), паттерн Strategy, super(), принципы OCP и LSP из SOLID.

Задача 31 (итерация по объекту: __iter__ и __next__). В зоопарке есть секция Section, и хочется перебирать животных в ней в цикле for — так же удобно, как список. Сейчас for animal in section: не работает. Реализуйте в классе Section магические методы __iter__ и __next__ так, чтобы по секции можно было итерироваться несколько раз подряд (это важно: __iter__ должен возвращать отдельный объект-итератор или генератор, а не self, иначе повторная итерация сломается). Создайте секцию с несколькими животными и переберите их в цикле for дважды подряд. Объясните, чем отличается итератор от итерируемого объекта, зачем нужны оба метода и почему list(section) должен возвращать список всех животных.

In [ ]:
class SectionIterator:
    def __init__(self, animals):
        self.animals = animals
        self.index = 0

    def __next__(self):
        if self.index >= len(self.animals):
            raise StopIteration
        animal = self.animals[self.index]
        self.index += 1
        return animal

class Section:
    def __init__(self, zone, curator):
        self.zone = zone
        self.curator = curator
        self.animals = []

    def add(self, animal):
        self.animals.append(animal)
        print(f"{animal.name} добавлен в секцию {self.zone}")

    def __iter__(self):
        return SectionIterator(self.animals)

    def info(self):
        print(f" Секция {self.zone}, куратор: {self.curator}")

class Animal:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    def info(self):
        return f" {self.name} - {self.species}"

    def __repr__(self):
        return f"Animal('{self.name}', '{self.species}')"

#Демонстрация

savanna = Section("A1", "Иван Иванов")
simba = Animal("Симба", "Лев")
raffa = Animal("Рафа", "Жираф")
zazu = Animal("Зазу", "Птица")

savanna.add(simba)
savanna.add(raffa)
savanna.add(zazu)

# итерация по секции
for animal in savanna:
    print(f"  {animal.info()}")

# ВТОРАЯ итерация по той же секции
for animal in savanna:
    print(f"  {animal.info()}")

animals_list = list(savanna)
print(f"Список: {animals_list}")
print(f"Длина: {len(animals_list)}")

"""
Чем отличается итератор от итерируемого объекта?
Итерируемый объект (Iterable, например Section) — это контейнер с данными.
Он умеет выдавать итератор через метод __iter__(), но сам не помнит позицию
в обходе. Итератор (Iterator, например SectionIterator) — это объект, который
помнит текущую позицию и умеет возвращать следующий элемент через __next__(),
а когда элементы кончаются бросает исключение StopIteration.

Зачем нужны оба метода?
__iter__ у итерируемого объекта создаёт новый итератор каждый раз, когда
начинается цикл for. Благодаря этому одну и ту же секцию можно перебирать
многократно, каждый цикл for получает свою собственную "закладку" с index=0.
__next__ у итератора отвечает за пошаговый обход: он возвращает следующий
элемент и сдвигает указатель. Когда элементы заканчиваются, он бросает сигнал для цикла for остановиться (StopIteration).
Почему list(section) возвращает список всех животных?
Встроенная функция list() в Python умеет работать с любым итерируемым объектом.
Она вызывает __iter__() для получения итератора, затем в цикле вызывает
__next__() до тех пор, пока не получит StopIteration, собирая все элементы
в список. Поскольку реализовали протокол итерации в Section, list(section)
автоматически начинает работать без единой дополнительной строчки кода.
"""

Задача 32 (индексация объекта: __getitem__ и __setitem__). В секции животные должны быть доступны по индексу — как элементы списка: section[0] возвращает первое животное, section[1] = animal заменяет второе. Реализуйте в классе Section магические методы __getitem__ и __setitem__. Продемонстрируйте, что теперь работают: section[0] — получение животного по индексу, section[1] = new_animal — замена животного, срезы section[0:2] — получение подсписка, for animal in section — итерация (за счёт __getitem__, даже без __iter__). Дополнительно реализуйте __delitem__ (удаление по индексу) и __contains__ (проверка animal in section). Объясните, как __getitem__ делает объект «похожим» на список или словарь и почему важно, чтобы он выбрасывал именно IndexError, а не какое-то своё исключение.

In [ ]:

    def __init__(self, name, species):
        self.name = name
        self.species = species

    def __repr__(self):
        return f"Animal('{self.name}', '{self.species}')"

    def __eq__(self, other):
        if isinstance(other, Animal):
            return self.name == other.name and self.species == other.species
        return False

class Section:
    def __init__(self, zone, curator):
        self.zone = zone
        self.curator = curator
        self.animals = []

    def add(self, animal):
        self.animals.append(animal)
        print(f"{animal.name} добавлен в секцию {self.zone}")

    def __getitem__(self, index):
        return self.animals[index]

    def __setitem__(self, index, animal):
        self.animals[index] = animal
        print(f" Животное по индексу {index} заменено на {animal.name}")

    def __delitem__(self, index):
        deleted = self.animals.pop(index)
        print(f"️ {deleted.name} удалён из секции")

    def __contains__(self, animal):
        return animal in self.animals

#Демонстрация
savanna = Section("A1", "Иван Иванов")
simba = Animal("Симба", "Лев")
raffa = Animal("", "Жираф")
zazu = Animal("Зазу", "Птица")

savanna.add(simba)
savanna.add(raffa)
savanna.add(zazu)

print(f"Первое животное: {savanna[0]}")
print(f"Второк животное: {savanna[1]}")

real_raffa = Animal("Рафа", "Жираф")
savanna[1] = real_raffa  # Заменяем животное с пустым именем

print(f"Первые два животного (срез [0:2]): {savanna[0:2]}")

del savanna[2]

print(f"Симба в секции? {simba in savanna}")   # ✅ True
print(f"Зазу в секции? {zazu in savanna}")     # ✅ False (удалили!)

animals_list = list(savanna)
print(f"Список: {animals_list}")

"""
Как __getitem__ делает объект «похожим» на список или словарь?
Реализация __getitem__ включает механизм обращения по индексу через
квадратные скобки []. Если делегировать эту операцию внутреннему списку
(self.animals[index]), объект автоматически получает поддержку отрицательных
индексов ([-1]) и срезов ([0:2]), так как встроенный список Python уже
умеет обрабатывать объекты типа slice, передаваемые в этот метод.

Почему важно выбрасывать именно IndexError, а не своё исключение?
Это критически важно для работы встроенных механизмов Python. Если у
объекта нет метода __iter__, Python использует протокол __getitem__ для
итерации: он последовательно запрашивает индексы 0, 1, 2... до тех пор,
пока не получит исключение IndexError. Это сигнал для интерпретатора,
что элементы закончились и цикл for нужно корректно завершить. Если
выбросить другое исключение (например, ValueError), цикл for воспримет
это как ошибку программы и аварийно завершится, а не как нормальное
окончание последовательности.
"""

Задача 33 (контекстный менеджер: __enter__ и __exit__). В зоопарке есть операции, которые нужно выполнять «аккуратно»: например, при работе с базой данных животных её нужно открыть в начале и обязательно закрыть в конце — даже если произошла ошибка. Реализуйте класс AnimalDatabase как контекстный менеджер: методы __enter__ (открывает базу и возвращает её) и __exit__ (закрывает базу, даже если было исключение). Продемонстрируйте использование через with AnimalDatabase() as db:. Дополнительно покажите, что происходит, если внутри with возникнет исключение, — база всё равно закроется. Объясните, зачем нужны контекстные менеджеры, как они связаны с конструкцией try/finally, что делают аргументы exc_type, exc_value, traceback в __exit__ и что нужно вернуть, чтобы «проглотить» исключение.

In [ ]:
class AnimalDatabase:
    def __init__(self):
        self.is_open = False
        self.data = {}

    def __enter__(self):
        print("Открываю базу данных...")
        self.is_open = True
        self.data = {"Симба": "Лев", "Рафа": "Жираф"}
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Закрываю базу данных")
        self.is_open = False

        if exc_type is not None:
            print(f"{exc_type.__name__}: {exc_value}")
        return False

    def get_animal(self, name):
        if not self.is_open:
            raise RuntimeError("База данных закрыта!")
        return self.data.get(name)

    def add_animal(self, name, species):
        if not self.is_open:
            raise RuntimeError("База данных закрыта!")
        self.data[name] = species
        print(f"Добавлен {name} ({species})")

#Демонстрация (нормальная работа)

with AnimalDatabase() as db:
    print(f"База открыта? {db.is_open}")
    db.add_animal("Зазу", "Птица")
    print(f"Симба: {db.get_animal('Симба')}")
print(f"После with база открыта? {db.is_open}")


#Демонстрация с ошибкой
try:
    with AnimalDatabase() as db:
        print(f"База открыта? {db.is_open}")
        db.add_animal("Тимон", "Сурикат")
        raise ValueError("Ошибка")
        db.add_animal("Пумба", "Бородавочник")  # ← Не выполнится
except ValueError as e:
    print(f"{e}")
print(f"База открыта? {db.is_open}")

""""
Зачем нужны контекстные менеджеры?
Контекстные менеджеры гарантируют корректное освобождение ресурсов
(закрытие файлов, соединений с БД, блокировок) даже при возникновении
исключений.

Что делают аргументы exc_type, exc_value, traceback в __exit__?
Если внутри with произошло исключение, Python передаёт в __exit__ три
аргумента: exc_type — класс исключения (например, ValueError),
exc_value — объект исключения с сообщением, traceback — объект трассировки
с информацией о стеке вызовов. Если исключений не было, все три аргумента
равны None.

Что нужно вернуть, чтобы «проглотить» исключение?
Если __exit__ возвращает True — исключение считается обработанным и не
пробрасывается дальше (программа продолжает работу). Если возвращает
False или None — исключение пробрасывается дальше после выполнения __exit__.
Это полезно для временного подавления ожидаемых ошибок (например, через
contextlib.suppress)."""

Задача 34 (паттерн Strategy + super()). В зоопарке есть разные способы расчёта суточного рациона: фиксированный (по норме), по весу (граммов на килограмм массы) и «диетический» (для животных на диете — половина от фиксированного). Реализуйте паттерн «Стратегия»: базовый класс FeedingStrategy с методом calculate(weight), и три наследника — FixedFeeding, WeightFeeding, DietFeeding. Каждый наследник переопределяет calculate. Дополнительно используйте super() в классе DietFeeding, чтобы вызвать реализацию FixedFeeding и уменьшить результат вдвое. Продемонстрируйте работу всех трёх стратегий через единый интерфейс. Объясните, что такое паттерн «Стратегия», зачем он нужен, как он убирает длинные цепочки if/elif и как super() помогает переиспользовать код родителя.

In [ ]:
class FeedingStrategy:
    def calculate(self, weight):
        raise NotImplementedError("Подклассы должны реализовать этот метод!")

#Стратегия 1
class FixedFeeding(FeedingStrategy):
    def calculate(self, weight):
        return 5.0 #фиксированное значение в кг

#Стратегия 2
class WeightFeeding(FeedingStrategy):
    def calculate(self, weight):
        return weight * 0.05  # 50г = 0.05кг на каждый кг массы

#Стратегия 3
class DietFeeding(FixedFeeding):
    def calculate(self, weight):
        fixed_amount = super().calculate(weight)
        return fixed_amount / 2


class Animal:
    def __init__(self, name, species, weight, feeding_strategy):
        self.name = name
        self.species = species
        self.weight = weight
        self.feeding_strategy = feeding_strategy

    def calculate_food(self):
        amount = self.feeding_strategy.calculate(self.weight)
        return f"{self.name} ({self.species}, {self.weight}кг): {amount:.2f} кг корма"

#Демонстрация
simba = Animal("Симба", "Лев", 200, FixedFeeding())
raffa = Animal("Рафа", "Жираф", 800, WeightFeeding())
zazu = Animal("Зазу", "Птица", 2, DietFeeding())

print(simba.calculate_food())
print(raffa.calculate_food())
print(zazu.calculate_food())

simba.feeding_strategy = DietFeeding()
print(simba.calculate_food())

"""
Что такое паттерн «Стратегия»?
Паттерн Strategy — это поведенческий паттерн проектирования, который
выносит семейство алгоритмов в отдельные классы (стратегии) и делает их
взаимозаменяемыми. Основной класс (контекст) не знает, какая именно
стратегия используется — он просто делегирует работу через единый интерфейс.

Зачем он нужен и как убирает if/elif?
Без Strategy все варианты алгоритмов собираются в одном методе через
длинные цепочки if/elif. Strategy решает эту
проблему, т.к каждый алгоритм является отдельным классом. Чтобы добавить новую
стратегию, мы просто создаём новый класс, не трогая основной код.
Основной класс остаётся стабильным, а поведение легко расширяется.

Как super() помогает переиспользовать код родителя?
super() предоставляет доступ к методам родительского класса. Вместо того
чтобы дублировать логику (например, расчёт фиксированного рациона в
DietFeeding), мы вызываем super().calculate(weight), получаем готовый
результат от родителя (FixedFeeding) и модифицируем его (делим на 2).
Если родитель изменит свою логику, наследник автоматически получит новую
версию — код не нужно править в двух местах.
"""

Задача 35 (принципы OCP и LSP из SOLID). В зоопарке появился новый класс животных — рептилии (Reptile). Если добавлять их через изменение базового класса Creature, это нарушит принцип OCP (Open/Closed Principle) — классы должны быть открыты для расширения, но закрыты для изменения. Дополнительно нужно проверить принцип LSP (Liskov Substitution Principle) — объект наследника должен без проблем заменять объект родителя. Реализуйте: базовый класс Creature с методом daily_food(); наследников Mammal, Bird; функцию process_creature(creature: Creature), которая работает с любым животным через родительский интерфейс. Добавьте Reptile без изменения существующего кода — это и есть OCP. Затем проверьте, что process_creature одинаково хорошо работает со всеми тремя типами, — это LSP. Объясните оба принципа и приведите примеры их нарушения (классический пример нарушения LSP: «квадрат — не прямоугольник», если сеттеры ширины и высоты меняют оба измерения).

In [ ]:
class Creature:
    def daily_food(self):
        raise NotImplementedError("Подклассы должны реализовать daily_food()")

    def describe(self):
        return f"Существо: {self.__class__.__name__}"


class Mammal(Creature):
    def __init__(self, name):
        self.name = name

    def daily_food(self):
        return 5.0

    def describe(self):
        return f" Млекопитающее {self.name}"


class Bird(Creature):
    def __init__(self, name):
        self.name = name

    def daily_food(self):
        return 1.0

    def describe(self):
        return f" Птица {self.name}"

#Добавляем рептилий
class Reptile(Creature):
    def __init__(self, name):
        self.name = name

    def daily_food(self):
        return 2.0

    def describe(self):
        return f" Рептилия {self.name}"

def process_creature(creature):
    print(f"{creature.describe()} суточный рацион: {creature.daily_food():.2f} кг")

#Демонстрация
simba = Mammal("Симба")
zazu = Bird("Зазу")
rex = Reptile("Рекс")  # ← Новый тип! process_creature о нём НЕ ЗНАЛ!

process_creature(simba)
process_creature(zazu)
process_creature(rex)

creatures = [simba, zazu, rex]

total_food = 0
for creature in creatures:
    total_food += creature.daily_food()

print(f"Общий рацион зоопарка: {total_food:.2f} кг")

"""
Принцип OCP (Open/Closed Principle):
 Это значит, что при появлении новых требований (например,
нового типа животного Reptile) добавляется новый код (новый класс),
а не правитсясуществующий (базовый класс Creature или функцию
process_creature).

Принцип LSP (Liskov Substitution Principle):
Объекты наследников должны быть заменяемы объектами родителя без
нарушения правильности программы. Если функция работает с родителем
(Creature), она должна работать и с любым наследником (Mammal, Bird,
Reptile).

Классический пример нарушения LSP — «квадрат и прямоугольник»:
Если Square наследуется от Rectangle и переопределяет set_width/set_height
так, что они меняют оба измерения, это нарушает ожидания функции,
которая считает ширину и высоту независимыми. Функция test_rectangle
ожидает, что после set_width(5) и set_height(4) площадь будет 20,
но у Square получится 16. Решение — сделать Square и Rectangle
независимыми наследниками общего родителя Shape, а не друг друга.
"""

 Уровень 6. Гуру (задачи 36–40)
Что изучается: объект как функция (__call__), разница между __new__ и __init__, паттерны Decorator и Adapter, принципы ISP и DIP из SOLID.

Задача 36 (объект как функция: __call__). В зоопарке есть служебный объект FoodCalculator, который должен уметь рассчитывать порцию для животного по его весу. Удобно, чтобы его можно было вызывать как функцию: calculator(weight), а не calculator.calculate(weight). Реализуйте в классе FoodCalculator магический метод __call__(self, weight), который возвращает порцию в граммах (например, weight * 50). Продемонстрируйте, что объект можно вызывать как функцию: calc = FoodCalculator(); calc(200). Дополнительно покажите, что такой объект можно передать туда, где ожидается функция-колбэк. Объясните, зачем нужен __call__, чем он полезен (функторы, декораторы-классы, колбэки с состоянием) и как он связан с понятием «функтор» в других языках.

In [ ]:
class FoodCalculator:
    def __init__(self, grams_per_kg=50):
        self.grams_per_kg = grams_per_kg
        self.call_count = 0

    def __call__(self, weight):
        self.call_count += 1
        result = weight * self.grams_per_kg
        print(f"[Вызов #{self.call_count}] {weight}кг × {self.grams_per_kg}г = {result}г")
        return result

# Демонстрация
calc = FoodCalculator()
result1 = calc(200)
result2 = calc(50)
result3 = calc(100)

print(f"\nВсего вызовов: {calc.call_count}")

#  Калькуляторы с разным коэффицентами
diet_calc = FoodCalculator(grams_per_kg=30)   # Диета — меньше порция
bulk_calc = FoodCalculator(grams_per_kg=80)   # Откорм — больше порция

diet_calc(100)
bulk_calc(100)

#Демонстрация 3
def process_weights(weights, calculator):
    results = []
    for weight in weights:
        results.append(calculator(weight))
    return results

weights = [50, 100, 150]
results = process_weights(weights, calc)
print(f"Результаты: {results}")


#Демонстрация 4
weights = [20, 40, 60]
portions = list(map(calc, weights))
print(f"Порции: {portions}")

"""
Зачем нужен __call__ и чем он полезен?
Метод __call__ позволяет вызывать экземпляр класса как обычную
функцию. Это объединяет преимущества ООП (хранение состояния, инкапсуляция)
и функционального программирования (удобный синтаксис вызова).

1. Функторы: Это объекты, которые ведут себя как функции, но имеют состояние.
   Обычная функция не может помнить количество своих вызовов или хранить
   настраиваемый коэффициент между запусками, а объект с __call__ — может.

2. Колбэки с состоянием: Такой объект можно передать в map(), filter(),
   sorted(key=...) или любую функцию, ожидающую колбэк. При этом объект
   сохраняет своё состояние между вызовами, что невозможно для простой функции.

3. Декораторы-классы: Класс с __call__ можно использовать как декоратор.
   Это позволяет параметризовать декоратор через __init__, что даёт больше
   гибкости, чем обычные функции-декораторы.

Связь с понятием «функтор» в других языках:
В C++ функтор — это класс с перегруженным operator(). В Java — класс,
реализующий функциональный интерфейс (например, Callable или Runnable).
В C# — делегат. Идея везде одна: объект, который можно вызвать как функцию.
Python реализует это наиболее элегантно через __call__, делая код чище.
"""

Задача 37 (__new__ vs __init__). В зоопарке есть класс Animal, и нужно понять, в каком порядке Python создаёт объект. Реализуйте в классе Animal оба метода — __new__ и __init__ — и добавьте в каждый отладочный вывод (print), чтобы увидеть порядок вызова. Продемонстрируйте, что __new__ вызывается до __init__ и отвечает за создание объекта, а __init__ — за его инициализацию. Дополнительно реализуйте __new__, который возвращает уже существующий объект (например, для кэширования животных по кличке и виду): два вызова Animal("Симба", "лев", 7) должны вернуть один и тот же объект. Объясните разницу между __new__ и __init__, зачем вообще нужен __new__ и почему «__new__ + кэш» легко приводит к странному поведению.

In [ ]:
class Animal:
    def __new__(cls, name, species, age):
        print(f"[__new__] Создаю объект для {name}...")
        instance = super().__new__(cls)
        print(f"[__new__] Объект создан! ID: {id(instance)}")
        return instance

    def __init__(self, name, species, age):
        print(f"[__init__] Инициализирую объект {name}...")
        self.name = name
        self.species = species
        self.age = age
        print(f"[__init__] Готово! {self.name} ({self.species}, {self.age} лет)")


# Демонстрация
simba = Animal("Симба", "Лев", 7)
print(f"\nID объекта: {id(simba)}\n")

In [ ]:
class CachedAnimal:
    _cache = {}

    def __new__(cls, name, species, age):
        cache_key = (name, species)
        print(f"[__new__] Проверяю кэш для ключа {cache_key}...")

        if cache_key in cls._cache:
            print(f"[__new__] Нашёл в кэше. Возвращаю существующий объект.")
            return cls._cache[cache_key]

        print(f"[__new__] Не нашёл в кэше. Создаю новый объект...")
        instance = super().__new__(cls)
        cls._cache[cache_key] = instance
        return instance

    def __init__(self, name, species, age):
        print(f"[__init__] Инициализирую {name} (возраст: {age})...")
        self.name = name
        self.species = species
        self.age = age

# Демонстраци
simba1 = CachedAnimal("Симба", "Лев", 7)
print(f"ID simba1: {id(simba1)}\n")

simba2 = CachedAnimal("Симба", "Лев", 10)
print(f"ID simba2: {id(simba2)}")

print(f"\nsimba1 is simba2? {simba1 is simba2}")
print(f"simba1.age = {simba1.age}")  # 10 — возраст изменился!
print(f"simba2.age = {simba2.age}")  # 10

"""
Разница между __new__ и __init__:
__new__ — это метод, который создает объект (выделяет память, возвращает
экземпляр класса). Он вызывается первым.
 __init__ — это метод, который инициализирует уже созданный объект.
Он вызывается вторым, после __new__.

Зачем нужен __new__?
Обычно переопределяют только __init__, потому что для большинства задач
достаточно инициализации. Но __new__ нужен в трёх случаях:
1. Неизменяемые типы (int, str, tuple) — значение нужно задать ДО создания,
   потому что после создания его нельзя изменить.
2. Паттерн Singleton — гарантировать, что существует только один экземпляр.
3. Кэширование — возвращать уже существующие объекты для экономии памяти.

Почему «__new__ + кэш» приводит к странному поведению?
Если __new__ возвращает уже существующий объект из кэша, Python ВСЁ РАВНО
вызывает __init__ для этого объекта. Это означает, что при втором вызове
Animal("Симба", "лев", 10) метод __init__ перезапишет атрибуты уже
существующего объекта (например, изменит возраст с 7 на 10).
"""

Задача 38 (паттерн Decorator). В зоопарке есть базовый класс ZooItem с методом get_info(). Нужно добавлять к объекту дополнительную информацию — например, «находится на карантине», «участвует в шоу», «редкий вид». При этом нельзя менять сам класс ZooItem. Реализуйте паттерн «Декоратор»: класс ItemDecorator, который содержит объект ZooItem внутри себя и имеет тот же метод get_info(), но добавляет к результату дополнительный текст. Создайте декораторы QuarantineDecorator и RareSpeciesDecorator, которые можно комбинировать: QuarantineDecorator(RareSpeciesDecorator(animal)). Продемонстрируйте работу, а также покажите, что смена порядка декораторов меняет порядок меток в выводе. Объясните, чем паттерн «Декоратор» отличается от наследования и почему он удобнее, когда нужно много комбинаций украшений (иначе пришлось бы создавать 2^n подклассов).

In [ ]:
class ZooItem:
    def __init__(self, name, species):
        self.name = name
        self.species = species

    def get_info(self):
        return f"{self.name} ({self.species})"

class ItemDecorator:
    def __init__(self, zooitem):
        self._zooitem = zooitem

    def get_info(self):
        return self._zooitem.get_info()

#Hа карантине
class QuarantineDecorator(ItemDecorator):
    def get_info(self):
        base_info = super().get_info()
        return f"{base_info} карантин!"

#Редкий вид
class RareSpeciesDecorator(ItemDecorator):
    def get_info(self):
        base_info = super().get_info()
        return f"{base_info} редкий вид!"

#Участвует в шоу»
class ShowDecorator(ItemDecorator):

    def get_info(self):
        base_info = super().get_info()
        return f"{base_info} участвует в шоу!"

#Демонстрация 1
print("Обычное животное:")
simba = ZooItem("Симба", "Лев")
print(simba.get_info())

#Демонстрация 2 (Один декоратор)
print("\nС одним декоратором (карантин)")
quarantine_simba = QuarantineDecorator(simba)
print(quarantine_simba.get_info())

#Демонстрация 3 (Комбинирование декораторов)
print("\n Комбинирование: карантин + редкий вид")
rare_quarantine_simba = QuarantineDecorator(RareSpeciesDecorator(simba))
print(rare_quarantine_simba.get_info())


print("\nСмена порядка декораторов")

order1 = QuarantineDecorator(RareSpeciesDecorator(simba))
print(f"Порядок 1: {order1.get_info()}")

order2 = RareSpeciesDecorator(QuarantineDecorator(simba))
print(f"Порядок 2: {order2.get_info()}")

#Демонстрация 4 (Три декоратора)
print("\n Все три декоратора сразу")
super_simba = ShowDecorator(QuarantineDecorator(RareSpeciesDecorator(simba)))
print(super_simba.get_info())


#Демонстрация 5
print("\nДинамическое добавление декораторов")
animal = ZooItem("Рафа", "Жираф")
print(f"Изначально: {animal.get_info()}")

animal = QuarantineDecorator(animal)
print(f"После карантина: {animal.get_info()}")

animal = ShowDecorator(animal)
print(f"После шоу: {animal.get_info()}")

animal = RareSpeciesDecorator(animal)
print(f"После редкого вида: {animal.get_info()}")

"""
Чем паттерн «Декоратор» отличается от наследования?
Декоратор использует композицию,
а наследование создаёт новые подклассы. Декоратор добавляет функциональность
динамически — во время выполнения программы, а наследование — статически,
на этапе написания кода.

Почему порядок декораторов важен?
При вызове
метода запрос идёт от внешнего декоратора к внутреннему, а результат
возвращается обратно, проходя через все слои. Внешний декоратор
добавляет свою метку последним, поэтому порядок декораторов определяет
порядок меток в выводе.
"""

Задача 39 (паттерн Adapter). В зоопарке появилась внешняя система учёта из другого зоопарка, которая возвращает данные о животных в другом формате — например, в виде словаря {"nickname": "Симба", "type": "лев", "years": 7}. А весь остальной код зоопарка работает с классом Animal, у которого поля name, species, age. Реализуйте паттерн «Адаптер»: класс AnimalAdapter, который принимает словарь и предоставляет интерфейс Animal — свойства name, species, age и метод info(). Продемонстрируйте, что адаптер можно использовать там, где ожидается Animal — например, передать его в функцию, которая работает с животным. Объясните, что такое паттерн «Адаптер», зачем он нужен, чем отличается от Facade и Decorator, и как он помогает интегрировать несовместимые интерфейсы без изменения существующего кода.

In [ ]:
class Animal:
    def __init__(self, name, species, age):
        self.name = name
        self.species = species
        self.age = age

    def info(self):
        return f"{self.name} - {self.species}, возраст {self.age} лет"

def process_animal(animal: Animal):
    print(f"  Имя: {animal.name}")
    print(f"  Вид: {animal.species}")
    print(f"  Возраст: {animal.age}")

#имитация внешней системы
def get_animal_from_external_system():
        return {
            "nickname": "Симба",
            "type": "лев",
            "years": 7}


class AnimalAdapter:

    def __init__(self, external_data: dict):
        self._data = external_data

    @property
    def name(self):
        return self._data["nickname"]

    @property
    def species(self):
        return self._data["type"]
    @property
    def age(self):
        return self._data["years"]

    def info(self):
        return f" {self.name} - {self.species}, возраст {self.age} лет"

#Демонстрация
print("Обычное животное")
simba = Animal("Симба", "Лев", 7)
process_animal(simba)


print("\n Внешние данные (другой формат)")
external_data = get_animal_from_external_system()
print(f"Данные из внешней системы: {external_data}")


print("\nИспользуем адаптер")
adapted_animal = AnimalAdapter(external_data)
print(f"Адаптированное животное: {adapted_animal.info()}")

process_animal(adapted_animal)

"""
Что такое паттерн «Адаптер»?
Адаптер — это структурный паттерн, который позволяет объектам с
несовместимыми интерфейсами работать вместе.
Зачем он нужен?
Он нужен для интеграции внешнего кода, старых библиотек или API с
другим форматом данных. Вместо переписывания всей системы под новый
формат создается адаптер.

Чем отличается от Facade и Decorator?
Adapter делает несовместимы интерфейсы совместимыми (переводит форматы).
Decorator добавляет функциональность, не меняя интерфейс (обёртка).
Facade упрощает сложный интерфейс, предоставляя одну кнопку вместо десяти.

Как помогает интегрировать несовместимые интерфейсы?
Адаптер принимает "чужой" объект и предоставляет методы/свойства,
которые ожидаются "нашей" системой. Внутри он преобразует имена полей
или форматы данных.
"""

Задача 40 (принципы ISP и DIP из SOLID). В зоопарке есть базовый класс ZooItem, и постепенно в нём накопились методы, которые нужны не всем объектам: feed() (только для животных), clean() (только для вольеров), show() (только для выставочных объектов). Это нарушает принцип ISP (Interface Segregation Principle) — клиент не должен зависеть от методов, которые он не использует. Разделите интерфейс на несколько маленьких: Feedable, Cleanable, Showable, и пусть каждый класс реализует только нужные. Дополнительно проверьте принцип DIP (Dependency Inversion Principle) — модули высокого уровня не должны зависеть от модулей низкого уровня; оба должны зависеть от абстракций. Реализуйте класс FeedingService, который зависит не от конкретных классов (Lion, Elephant), а от абстракции Feedable (или от Protocol). Продемонстрируйте, что можно легко добавить новый тип животного (например, Reptile, реализующий только Feedable) без изменения FeedingService. Объясните оба принципа, приведите примеры их нарушения и расскажите, почему DIP помогает тестировать код (легко подменять зависимости на mock).

In [ ]:
from typing import Protocol

class Feedable(Protocol):
    def feed(self) -> str: ...


class Cleanable(Protocol):
    def clean(self) -> str: ...


class Showable(Protocol):

    def show(self) -> str: ...


class Lion:

    def feed(self) -> str:
        return "Лев съел 5 кг мяса"

    def show(self) -> str:
        return "Лев рычит для посетителей"


class Enclosure:
    def clean(self) -> str:
        return "Вольер вымыт"


class Reptile:

    def feed(self) -> str:
        return "Рептилия съела сверчка"

class FeedingService:
    def feed_animal(self, animal: Feedable):
        print(f" Запись в журнале кормления: {animal.feed()}")

print("Работа с существующими классами")
feeding_service = FeedingService()

lion = Lion()
feeding_service.feed_animal(lion)

reptile = Reptile()
feeding_service.feed_animal(reptile)


#Фейковый класс
class MockAnimal:
    def feed(self) -> str:
        return " MOCK: Животное накормлено (тестовые данные)"

mock_animal = MockAnimal()
feeding_service.feed_animal(mock_animal)


"""
Принцип ISP (Interface Segregation Principle):
Клиент не должен зависеть от методов, которые он не использует. Вместо
одного большого интерфейса (например, ZooItem с методами feed, clean,
show) лучше создать несколько маленьких, специализированных интерфейсов
(Feedable, Cleanable, Showable).
Пример нарушения: класс Enclosure вынужден реализовывать пустые методы
feed() и show(), возвращая NotImplementedError, только потому что они
объявлены в базовом классе ZooItem.

Принцип DIP (Dependency Inversion Principle):
Модули высокого уровня (например, FeedingService) не должны зависеть от
модулей низкого уровня (конкретных классов Lion, Reptile). Оба должны
зависеть от абстракций (протокола Feedable).
Пример нарушения: FeedingService содержит цепочку if/elif, проверяющую
type(animal), что заставляет менять код сервиса при добавлении нового
вида животного.

Почему DIP помогает тестировать код?
Поскольку высокоуровневый модуль зависит от абстракции, а не от
конкретной реализации, во время юнит-тестирования можно легко
подменить реальную зависимость на Mock-объект (заглушку), который
реализует тот же интерфейс. Это позволяет тестировать логику сервиса
изолированно, без необходимости создавать реальные сложные объекты
или подключаться к базе данных.
"""